In [ ]:
# players: list of dicts {'name': str, 'rank': int, 'position': list of ['F', 'D']}
# rank options: [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0]

players = [
    { "name": "Cole", "rank": 1.0, "positions": ["F"]},
    { "name": "Erik", "rank": 1.0, "positions": ["F"]},
    { "name": "Kendra", "rank": 1.0, "positions": ["D"]},
    { "name": "Henry", "rank": 1.5, "positions": ["D"]},
    { "name": "Viktor", "rank": 1.5, "positions": ["F"]},
    { "name": "Sarah", "rank": 1.5, "positions": ["F"]},
    { "name": "Mikael", "rank": 2.0, "positions": ["F"]},
    { "name": "Josie", "rank": 2.0, "positions": ["F"]},
    { "name": "Anniina", "rank": 2.5, "positions": ["F"]},
    { "name": "Brody", "rank": 2.5, "positions": ["D","F"]},
    { "name": "Hilary", "rank": 3.0, "positions": ["F"]},
    { "name": "Jake", "rank": 3.5, "positions": ["F"]},
    { "name": "Dominik", "rank": 3.5, "positions": ["D","F"]},
    { "name": "Kayla", "rank": 4.0, "positions": ["F"]},
    { "name": "Aleksander", "rank": 4.5, "positions": ["F"]},
    { "name": "Megan", "rank": 4.5, "positions": ["D", "F"]},
    { "name": "Blayre", "rank": 5.0, "positions": ["F"]},
    { "name": "Cale", "rank": 5.0, "positions": ["D"]},
    { "name": "Adam", "rank": 5.0, "positions": ["D"]},
    { "name": "Quinn", "rank": 6.0, "positions": ["D"]},
    { "name": "Niklas", "rank": 6.0, "positions": ["F"]},
]

n_players = len(players)

### Manually Specify Values

In [13]:
n_teams = 2
min_forwards_per_team = 3
min_defenders_per_team = 2

# Weights indicating how important each soft constraint is
per_rank_tier_balance_weight = 20
rank_sum_balance_weight = 5
forwards_balance_weight = 10
defenders_balance_weight = 15

# Filepath for Excel spreadsheet with player data
FILEPATH = "players.xlsx"

### Fixed Values

In [14]:
team_size_balance_weight = 10
min_forwards_met_weight = 20
min_defenders_met_weight = 20

### Read in Excel Sheet

In [ ]:
import pandas as pd

def get_player(row)-> dict:
    return {
        "name": row.Name,
        "rank": row.Rank,
        "positions": set([pos[0].upper() for pos in [row.Position_1, row.Position_2] if (pos and str(pos) and str(pos) != "nan")])
    }

def get_players(filepath:str) -> list[dict]:
    df = pd.read_excel(filepath)
    df = df.rename(columns=lambda x: x.replace(' ', '_'))
    return [get_player(row) for row in df.itertuples()]

def get_rank_dict(players:list[dict]) -> dict[float, int]:
    rank_dict = {}
    for player in players:
        rank = player["rank"]
        if rank not in rank_dict:
            rank_dict[rank] = 0
        rank_dict[rank] += 1
    return rank_dict


players = get_players(FILEPATH)
n_players = len(players)
rank_dict = get_rank_dict(players)

In [ ]:
"""
SOURCES USED:
- https://cpmpy.readthedocs.io/en/latest/modeling.html
- Used Claude AI to translate desired rules into cpmpy constraints for this assignment/minimization problem
  Manually reviewed generated code to verify correctness
"""

import cpmpy as cp

# x = assignment table (n_players, n_teams)
x = cp.boolvar(shape=(n_players, n_teams), name="x")  # x[p,t] = 1 if player p on team t

# Initialize the model
model = cp.Model()

# HARD CONSTRAINT: each player on exactly one team
model += (x.sum(axis=1) == 1)

# HARD: flex player can only be a forward OR a defender in a game
is_flex = [p["positions"] == set(["F", "D"]) for p in players]
y = cp.boolvar(shape=n_players, name="is_forward")  # y = tracks if player is assigned as a forward (1 = counts as F)
for p in range(n_players):
    if not is_flex[p]:
        model += (y[p] == (1 if "F" in players[p]["positions"] else 0))

# Track soft constraint penalties
penalties = []

# SOFT: teams are evenly sized (only differ by at most 1 player)
team_sizes = x.sum(axis=0)
size_diff = cp.max(team_sizes) - cp.min(team_sizes)
penalties.append(team_size_balance_weight * size_diff)

# SOFT: forward/defender minimums met
for t in range(n_teams):
    forwards_on_team = cp.sum(y[p] * x[p, t] for p in range(n_players))
    defenders_on_team = cp.sum((1 - y[p]) * x[p, t] for p in range(n_players))
    penalties.append(min_forwards_met_weight * cp.max([min_forwards_per_team - forwards_on_team, 0]))
    penalties.append(min_defenders_met_weight * cp.max([min_defenders_per_team - defenders_on_team, 0]))

# SOFT: similar number of forwards per team
forwards_per_team = [cp.sum(y[p] * x[p, t] for p in range(n_players)) for t in range(n_teams)]
forward_diff = cp.max(forwards_per_team) - cp.min(forwards_per_team)
penalties.append(forwards_balance_weight * forward_diff)

# SOFT: similar number of defenders per team
defenders_per_team = [cp.sum((1 - y[p]) * x[p, t] for p in range(n_players)) for t in range(n_teams)]
defender_diff = cp.max(defenders_per_team) - cp.min(defenders_per_team)
penalties.append(defenders_balance_weight * defender_diff)

# SOFT: per-rank-tier loop (avoid having a team be all 1's and 6's and another all 3-4's)
rank_values = sorted(rank_dict.keys())
for rank in rank_values:
    # indices of players at this rank
    idxs = [p for p in range(n_players) if players[p]["rank"] == rank]
    # how many players at this rank end up on each team
    counts_per_team = [cp.sum(x[p, t] for p in idxs) for t in range(n_teams)]
    # diff in counts of players per rank between teams
    tier_diff = cp.max(counts_per_team) - cp.min(counts_per_team)
    # penalty for imbalanced player rank distribution
    penalties.append(per_rank_tier_balance_weight * tier_diff)

# SOFT: overall rank-sum balance
ranks = [int(p["rank"] * 2) for p in players]   # 1.0 -> 2, 1.5 -> 3, etc. (scaled to ints to be compatible with Google OR-tools CP-SAT)
team_rank_totals = [cp.sum(ranks[p] * x[p, t] for p in range(n_players)) for t in range(n_teams)]
rank_diff = cp.abs(team_rank_totals[0] - team_rank_totals[1])
penalties.append(rank_sum_balance_weight * rank_diff)

# Minimize sum of penalties
model.minimize(cp.sum(penalties))

lines = []
if model.solve():
    for t in range(n_teams):
        lines.append(f"Team {t+1}:")
        team = { "F": {}, "D": {} }
        for p in range(n_players):
            if x[p, t].value():
                role = "F" if y[p].value() else "D"
                flex_tag = " (flex)" if is_flex[p] else ""
                player_rank = players[p]['rank']
                if player_rank not in team[role]:
                    team[role][player_rank] = []
                team[role][player_rank].append({"name": players[p]["name"], "role": f"{role}{flex_tag}"})
        for pos in team.keys():
            rank_values = sorted(team[pos].keys())
            i = 1
            for rank in rank_values:
                for p in team[pos][rank]:
                    lines.append(f"  {i}. {p['role']} - {p['name']} ({rank})")
                    i = i + 1
            lines.append("")
        lines.append("")
else:
    lines.append("Even the relaxed model is infeasible — check that x.sum(axis=1)==1 is satisfiable at all")


for line in lines:
    print(line)


Team 1:
  1. F - Erik (1.0)
  2. F - Viktor (1.5)
  3. F - Sarah (1.5)
  4. F - Josie (2.0)
  5. F - Anniina (2.5)
  6. F - Jake (3.5)
  7. F - Aleksander (4.5)

  1. D - Kendra (1.0)
  2. D - Cale (5.0)
  3. D - Adam (5.0)
  4. D - Quinn (6.0)


Team 2:
  1. F - Cole (1.0)
  2. F - Mikael (2.0)
  3. F - Hilary (3.0)
  4. F - Kayla (4.0)
  5. F - Blayre (5.0)
  6. F - Niklas (6.0)

  1. D - Henry (1.5)
  2. D (flex) - Brody (2.5)
  3. D (flex) - Dominik (3.5)
  4. D (flex) - Megan (4.5)


